## Imports

In [233]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from imblearn.under_sampling import RandomUnderSampler
import pickle

## Utils

In [234]:
def get_data_root():
  '''
    Ritorna il percorso della cartella contenente i dati in base all'ambiente di esecuzione.
  '''
  try:
      import google.colab
      from google.colab import drive

      try:
          drive.mount("/content/drive", force_remount=True)
          return "/content/drive/MyDrive/ColabContent/Data_analytics"
      except Exception:
          print("Drive non montabile")
          return "/content"

  except ImportError:
      return "../../data"

print(get_data_root())

../../data


## Global variables

In [ ]:
DATA_ROOT = get_data_root()
DATASET_PATH = f"{DATA_ROOT}/Dataset2526/train.csv"
SEED = 42
MI_THRESHOLD = 0.01
CORR_THRESHOLD = 0.95

# Preprocessing

In [236]:
df = pd.read_csv(DATASET_PATH)

df, df_test = train_test_split(df, test_size=0.2, random_state=SEED)
df_val, df_test = train_test_split(df_test, test_size=0.5, random_state=SEED)

df.to_csv(f"{DATA_ROOT}/train_raw.csv", index=False)
df_val.to_csv(f"{DATA_ROOT}/val_raw.csv", index=False)
df_test.to_csv(f"{DATA_ROOT}/test_raw.csv", index=False)

KeyboardInterrupt: 

### Feature selection

In [ ]:
# Drop colonne con troppi nan
threshold = 0.25
missing_percent = df.isnull().mean()
cols_to_drop_nan = missing_percent[missing_percent > threshold].index
df_cleaned = df.drop(columns=cols_to_drop_nan)

print(f"Droppate {len(cols_to_drop_nan)} colonne con nan > {threshold*100}%")

Droppate 56 colonne con nan > 25.0%


In [ ]:
# Drop colonne poco significative
cols_to_drop_manual = [
    # Testo
    'loan_title',

    # Post-Emissione
    'recoveries_cash', 'collection_recovery_fee', 'total_received_late_fees',
    'loan_status_current_code',
]

df_cleaned = df_cleaned.drop(columns=[c for c in cols_to_drop_manual if c in df_cleaned.columns])

print(f"Droppate {len(cols_to_drop_manual)} colonne poco significative")

Droppate 5 colonne poco significative


### Gestione valori mancanti

In [ ]:
medians = df_cleaned.select_dtypes(include=[np.number]).median().to_dict()
modes = {col: df_cleaned[col].mode()[0] for col in df_cleaned.select_dtypes(include=['object']).columns}

# Mediana per feature numeriche
for col, val in medians.items():
    df_cleaned[col] = df_cleaned[col].fillna(val)
    
# Moda per feature categoriche
for col, val in modes.items():
    df_cleaned[col] = df_cleaned[col].fillna(val)

### Gestione date

In [ ]:
# Map per employment_length
emp_len_map = {
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}
df_cleaned['borrower_profile_employment_length'] = df_cleaned['borrower_profile_employment_length'].map(emp_len_map)

# Trasformazione delle date
df_cleaned['loan_issue_date'] = pd.to_datetime(df_cleaned['loan_issue_date'])
df_cleaned['credit_history_earliest_line'] = pd.to_datetime(df_cleaned['credit_history_earliest_line'])

# Calcolo esperienza finanziaria debitore
df_cleaned['months_credit_history'] = (df_cleaned['loan_issue_date'].dt.year - df_cleaned['credit_history_earliest_line'].dt.year) * 12 + \
                              (df_cleaned['loan_issue_date'].dt.month - df_cleaned['credit_history_earliest_line'].dt.month)

# Estrazione mese/anno delle date
date_cols = ['loan_issue_date', 'credit_history_earliest_line', 'last_payment_date', 'last_credit_pull_date']
for col in date_cols:
    if col in df_cleaned.columns:
        date_dt = pd.to_datetime(df_cleaned[col])
        df_cleaned[col + '_year'] = date_dt.dt.year
        df_cleaned[col + '_month'] = date_dt.dt.month

df_cleaned = df_cleaned.drop(columns=date_cols, errors='ignore')

/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_23280/3245697033.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cleaned['loan_issue_date'] = pd.to_datetime(df_cleaned['loan_issue_date'])
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_23280/3245697033.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cleaned['credit_history_earliest_line'] = pd.to_datetime(df_cleaned['credit_history_earliest_line'])
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_23280/3245697033.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_dt = p

### Gestione outlier

In [ ]:
df_out = df_cleaned.copy()
clipping_bounds = {}
num_cols = df_out.select_dtypes(include=[np.number]).columns
fico_cols = [c for c in df_out.columns if 'fico' in c.lower()]
perc_cols = [
  'revolving_utilization', 'bankcard_utilization', 'installment_utilization',
  'overall_utilization', 'bankcard_util_gt_75_ratio', 'tradelines_never_delinquent_ratio',
  'loan_contract_interest_rate'
]

for col in num_cols:
  Q1 = df_out[col].quantile(0.25)
  Q3 = df_out[col].quantile(0.75)
  IQR = Q3 - Q1

  # Applichiamo il clipping solo se l'IQR è maggiore di 0
  if IQR > 0:
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_out[col] = df_out[col].clip(lower=lower_bound, upper=upper_bound)
  # Altrimenti solo valori estremi
  else:
    upper_bound = df_out[col].quantile(0.95)
    df_out[col] = df_out[col].clip(upper=upper_bound)
  
  clipping_bounds[col] = (lower_bound, upper_bound)

# Gestione valori FICO >850
for col in fico_cols:
  df_out[col] = df_out[col].clip(upper=850)

# Gestione percentuali >100%
for col in perc_cols:
  if col in df_out.columns:
    # Check se percentuale 0-1 o 0-100
    if df_out[col].max() > 1:
      df_out[col] = df_out[col].clip(upper=100)
    else:
      df_out[col] = df_out[col].clip(upper=1.0)

### Drop colonne con alta correlazione

In [ ]:
corr_matrix = df_out.select_dtypes(include=[np.number]).corr().abs()

# Selezine triangolo superiore
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [column for column in upper.columns if any(upper[column] > CORR_THRESHOLD)]
#df_out = df_out.drop(columns=to_drop_corr)

print(f"Rimosse {len(to_drop_corr)} colonne per alta correlazione: {to_drop_corr}")

Rimosse 8 colonne per alta correlazione: ['loan_portfolio_total_funded', 'investor_side_funded_amount', 'outstanding_principal_investor_side', 'total_payment_investor_side', 'revolving_tradelines_balance_gt_0', 'satisfactory_accounts', 'total_high_credit_limit', 'credit_history_earliest_line_year']


### Drop colonne con varianza 0

In [ ]:
# Drop colonne con varianza 0
cols_to_drop_const = [col for col in df_out.columns if df_out[col].nunique() <= 1]
df_out = df_out.drop(columns=cols_to_drop_const)

print(f"Droppate {len(cols_to_drop_const)} colonne con varianza 0")

Droppate 7 colonne con varianza 0


### Drop colonne con mutual information

In [ ]:
y_mi = df_out['grade'].map({'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6})
X_mi = df_out.select_dtypes(include=[np.number]).drop(columns=['grade'], errors='ignore')

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=SEED)
mi_series = pd.Series(mi_scores, index=X_mi.columns)
low_info_cols = mi_series[mi_series < MI_THRESHOLD].index.tolist()

df_out = df_out.drop(columns=low_info_cols)

print(f"Droppate {len(low_info_cols)} colonne con bassa informazione")

Droppate 21 colonne con bassa informazione


### Encoding

In [ ]:
# Target
grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df_out['grade'] = df_out['grade'].map(grade_map)

# Colonne categoriche
numeric_features = df_out.drop(columns=['grade']).select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_out.drop(columns=['grade']).select_dtypes(include=['object']).columns.tolist()

# Dizionari per salvare gli oggetti
oh_encoders = {}
l_encoders = {}

for col in categorical_features:
    unique_vals = df_out[col].dropna().unique()

    if len(unique_vals) < 15:
        # One-hot
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
        ohe.fit(df_out[[col]])
        encoded_data = ohe.transform(df_out[[col]])
        column_names = [f"{col}_{cat}" for cat in ohe.categories_[0][1:]]

        encoded_df = pd.DataFrame(encoded_data, columns=column_names, index=df_out.index)
        df_out = pd.concat([df_out.drop(col, axis=1), encoded_df], axis=1)

        oh_encoders[col] = ohe

    else:
        # LabelEncoder
        le = LabelEncoder()
        df_out[col] = le.fit_transform(df_out[col].astype(str))

        l_encoders[col] = le

## Salvataggio asset

In [ ]:
final_columns = df_out.drop(columns=['grade']).columns.tolist()

with open('preprocessing_assets.pkl', 'wb') as f:
    pickle.dump({
        'final_features': final_columns,
        'numeric_features': numeric_features, 
        'categorical_features': categorical_features,
        'clipping_bounds': clipping_bounds,
        'oh_encoders': oh_encoders,
        'l_encoders': l_encoders,
        'medians': medians,
        'modes': modes,
        'emp_len_map': emp_len_map,
        'grade_map': grade_map
    }, f)

## Bilanciamento

In [ ]:
X = df_out.drop(columns=['grade'])
y = df_out['grade']

rus = RandomUnderSampler(random_state=SEED)
X_res, y_res = rus.fit_resample(X, y)

df_balanced = pd.DataFrame(X_res, columns=X.columns)
df_balanced['grade'] = y_res

print("Distribuzione dopo RandomUnderSampler:")
print(y_res.value_counts())

Distribuzione dopo RandomUnderSampler:
grade
0    4821
1    4821
2    4821
3    4821
4    4821
5    4821
6    4821
Name: count, dtype: int64


## Salvataggio train set preprocessato

In [ ]:
df_balanced.to_csv(f"{DATA_ROOT}/train_processed.csv", index=False)

## Applicazione preprocessing a validation e test set

In [ ]:
def preprocess(df_input, assets_path='preprocessing_assets.pkl'):
    """
    Versione corretta: estrae le componenti temporali e gestisce i NaN 
    prima dell'allineamento finale.
    """
    with open(assets_path, 'rb') as f:
        assets = pickle.load(f)
    
    df = df_input.copy()

    # 1. Mapping emplyment_length
    if 'borrower_profile_employment_length' in df.columns:
        df['borrower_profile_employment_length'] = df['borrower_profile_employment_length'].map(assets['emp_len_map'])

    # 2. Imputazione valori mancanti
    for col in assets['numeric_features']:
        if col in df.columns:
            df[col] = df[col].fillna(assets['medians'].get(col, 0)) 
    for col in assets['categorical_features']:
        if col in df.columns:
            df[col] = df[col].fillna(assets['modes'].get(col, 'Unknown'))

    # 3. Gestione date
    date_cols = ['loan_issue_date', 'last_payment_date', 'last_credit_pull_date', 'credit_history_earliest_line']
    
    for col in date_cols:
        if col in df.columns:
            dt_series = pd.to_datetime(df[col], errors='coerce')
            df[f"{col}_year"] = dt_series.dt.year
            df[f"{col}_month"] = dt_series.dt.month

    # 4. Feature engineering
    if 'loan_issue_date' in df.columns and 'credit_history_earliest_line' in df.columns:
        issue_dt = pd.to_datetime(df['loan_issue_date'], errors='coerce')
        hist_dt = pd.to_datetime(df['credit_history_earliest_line'], errors='coerce')
        df['months_credit_history'] = (issue_dt.dt.year - hist_dt.dt.year) * 12 + (issue_dt.dt.month - hist_dt.dt.month)

    # 5. Clipping outlier
    bounds = assets.get('clipping_bounds', {})
    for col, (low, high) in bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=low, upper=high)

    # 6. ENCODING
    # One-Hot Encoding
    for col, ohe in assets['oh_encoders'].items():
        if col in df.columns:
            encoded = ohe.transform(df[[col]])
            names = [f"{col}_{cat}" for cat in ohe.categories_[0][1:]]
            encoded_df = pd.DataFrame(encoded, columns=names, index=df.index)
            df = pd.concat([df.drop(col, axis=1), encoded_df], axis=1)

    # Label Encoding
    for col, le in assets['l_encoders'].items():
        if col in df.columns:
            df[col] = df[col].astype(str).map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

    # 7. Allineamento finale
    final_cols = assets['final_features']
    
    for col in final_cols:
        if col not in df.columns:
            df[col] = 0 
    
    # Solo le colonne del training
    df_out = df[final_cols].copy()
    
    # Ultimo controllo NaN 
    for col in df_out.columns:
        if col in assets['medians']:
            df_out[col] = df_out[col].fillna(assets['medians'][col])
        else:
            df_out[col] = df_out[col].fillna(0)

    # Gestione Target
    if 'grade' in df.columns:
        df_out['grade'] = df['grade'].map(assets['grade_map'])
    
    return df_out

In [ ]:
df_val_processed = preprocess(df_val)
df_test_processed = preprocess(df_test)

df_val_processed.to_csv(f"{DATA_ROOT}/val_processed.csv", index=False)
df_test_processed.to_csv(f"{DATA_ROOT}/test_processed.csv", index=False)

/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_23280/1886316444.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt_series = pd.to_datetime(df[col], errors='coerce')
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_23280/1886316444.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt_series = pd.to_datetime(df[col], errors='coerce')
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_23280/1886316444.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt_series = pd.to_datetime(df[col], errors='coerce')
/var/folders/93/wjn3tmxx3694z231t8